## Information extraction (IE) via LLMs and tool calling

**Information extraction** (IE) is often defined as the automated process of identifying and extracting specific, structured pieces of information from unstructured or semi-structured data sources, such as text documents, emails, reviews or medical recods (see [here](https://www.width.ai/post/emr-data-extraction) examples of such documents).
The primary goal is to transform human-readable, free-form text into a structured, machine-readable format, such as a database table or a knowledge graph, that computers can easily process and analyze.

Information extraction can be seamlessly done using LLMs that support ***tool / function calling***.

Tool calling is about providing an LLM with ability to communicate with external tools. ***Tools*** are defined as utilities (APIs, data bases, search engines, etc) designed to be called by a model: their inputs are designed to be generated by models, and their outputs are designed to be passed back to models.

Tool calling is well explained in [OpenAI's documentation](https://platform.openai.com/docs/guides/function-calling) and the figure below, taken from that documentation, offers a nice illustration of function (tool) calling workflow:

<img src="https://cdn.openai.com/API/docs/images/function-calling-diagram-steps.png" width="470" height="630" alt="Illustration of function calling workflow">

### Our task: information extraction from customer reviews

We will use tool-calling features of an open-weights LLM available via Groq and LangChain's integration with Groq, to extract structured information from text of customer reviews.

Note: LangChain supports tool-calling for LLMs of several providers. A list of supported LLMs is available, for example, [here](https://docs.langchain.com/oss/python/integrations/chat).

We start with the regular setup steps: loading the required packages and access tokens.

In [ ]:
from os import getenv
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

if not getenv("GROQ_API_KEY"):
    print("GROQ_API_KEY is necessary for running an LLM in the code below. Please include it in the .env file")

RAND_STATE = 9

## Schema for information extraction

Information extraction (IE) in LangChain relies on two key components:

* **Schema**: defines what pieces of information we want to extract from the text. For example, we may want to extract the expressed sentiment, the language of the text, people and/or places mentioned in the text, etc.
* **Extraction prompt**: instructs the model how to extract information (specified in the schema) from the text; optionally, it may include examples to improve IE through few-shot learning

Both the extraction prompt and the schema are passed to an LLM. The LLM does the information extraction, following the instructions from the prompt and the meta-data embedded in the schema. It then creates and returns a JSON object aligned with the schema; LangChain takes care of parsing the resulting JSON into an instance of the schema python class.

### Defining data schema in Pydantic

[Pydantic](https://docs.pydantic.dev/latest/) is a Python library used for data validation and parsing.
In the following we will familiarise with the pydantic elements that are required for defining the schema of the structured output we want an LLM to produce.

`BaseModel`
* This is the core class we inherit from when defining a data schema for the structured output. Any class that inherits from BaseModel is recognized by Pydantic (and LangChain) as a data structure definition.

* Why it is relevant?

  1) **Type checking / coercion**. It automatically checks the type of the extracted data. If the LLM returns "25" for an int field, Pydantic will coerce it to the integer 25. If the LLM returns "twenty-five", Pydantic will throw a validation error, which LangChain uses to prompt the LLM to retry.
  2) **JSON Schema Generation**. Pydantic can easily convert a Python class into a standard JSON Schema (via `model_json_schema()` method). This JSON Schema is what LangChain uses to define a tool to be passes to the LLM (via function calling).

In case of using such data model in LangChain to define the expected structured output, it is recommended to add a doc string stating the purpose of the data model (schema).

In [ ]:
from pydantic import BaseModel

class AppReview(BaseModel):
  """Information about a mobile app review."""
  pass

Next, pydantic allows us to define the expected data type for each field

In [ ]:
class AppReview(BaseModel):
  """Information about a mobile app review."""

  app_opinion: list[str]
  problems: list[str]
  sentiment: str
  satisfaction: int

It also allows us to define meta-data for each field in the schema, which is highly relevant as via meta-data we are communicating additional instructions to the LLM.

The `Field` function is used to add metadata to the schema fields. Its key arguments include:
* `description`: This is key argument as it tells the LLM exactly what information to extract for this field. A good description is often the difference between a poor and a precise extraction.
* `default` or `default_factory`: Sets a default value (or creates an empty instance of the given type) if the field is optional and not present in the input. It is generally recommended to make optional (and with a default value) all the fields corresponding to the data to be directly extracted from the text, to prevent LLM from halycinating to generate a field value when there is none in the input content.
* `examples`: One can provide a list of expected examples for a field, which further guides the LLM on the field's format and content. Examples are espcially useful for specifying date/time formats or any other conventions for respresenting data.



In [ ]:
from pydantic import Field
from typing import Optional


class AppReview(BaseModel):
    """Information about a mobile app review."""

    app_opinion: Optional[list] = Field(
        default_factory=list,
        description="The terms used in the review to describe the mobile app and its features",
        examples=[['well designed', 'very useful']]
    )

    problems: Optional[list] = Field(
        default_factory=list,
        description="Any problem that the user mentions in the review regarding the app",
        examples=[['blinking screen', 'annoying ads']]
    )

    sentiment: str = Field(
        description="The sentiment expressed in the review. It can be one of the following: Positive, Negative, Neutral, or Mixed",
        examples=['Positive']
    )

    satisfaction: int = Field(
        ge=0,
        le=5,
        description="""
        How satisfied the user seems to be with the app, on the integer scale from 0 to 5:
        0 = completely dissatisfied, 3 = mixed or average, 5 = completely satisfied.""",
        examples=[3]
    )

Finally, we can further advance the data structure by using python enums to specify allowed values for some of the fields, in this case, for the sentiment expressed in the review:

In [ ]:
from enum import Enum
from pydantic import BaseModel, Field
from typing import Optional

class Sentiment(Enum):
    NEGATIVE = "Negative"
    NEUTRAL = "Neutral"
    POSITIVE = "Positive"
    MIXED = "Mixed"

class AppReview(BaseModel):
    """Information about a mobile app review."""

    app_opinion: Optional[list] = Field(
        default_factory=list,
        description="The terms used in the review to describe the mobile app and its features",
        examples=[['well designed', 'very useful']]
    )

    problems: Optional[list] = Field(
        default_factory=list,
        description="Any problem that the user mentions in the review regarding the app",
        examples=[['blinking screen', 'annoying ads']]
    )

    sentiment: Sentiment = Field(
        description="The overall sentiment expressed in the review.",
        examples=[Sentiment.POSITIVE]
    )

    satisfaction: int = Field(
        ge=0,
        le=5,
        description="""
        How satisfied the user seems to be with the app, on the integer scale from 0 to 5:
        0 = completely dissatisfied, 3 = mixed or average, 5 = completely satisfied""",
        examples=[3]
    )

To learn more about Pydantic, you may want to:
* start from [this video](https://www.youtube.com/watch?v=XIdQ6gO3Anc), as it offers an excellent introduction to this python library for data typing and validation.
* try [this video tutorial](https://youtu.be/i4jespFbA1c?si=mjKneeBQdlDcQGll) and the accompanying [series of blog posts](https://www.bugbytes.io/posts/introduction-to-pydantic/) to  explore further pydantic's features for data and code validation.


### General recommendations for defining an IE schema

The following are recommendations (best practices) to follow when defining data schema to be used in the IE task:

* *The schema and the attributes it includes should be well documented*. The information contained in the documentation is sent to the LLM and allows for improving the quality of information extraction. This is exemplified in the code above with the doc-string describing the AppReview class as well as descriptions associated to each class attribute.

* Use `Optional` (from python's `typing` module) for a field when a piece of information might not exist in the source text and leaving it empty/null is better than letting an LLM to guess / invent the value.

* Keep the field required when every record should have a value, either because it is always present or because it is supposed to be computed/inferred for every item (e.g., sentiment).


### Back to the original task: the use of LLMs for IE from text

Now that we have defined a data schema for the structured output the LLM should produce, we create an extraction prompt. As before, we are creating a prompt for a chat model and thus specify it as a list of (system and user) messages:

In [ ]:
ie_system_msg = """
You are an expert information extraction agent specialized in analyzing mobile app reviews.
Extract literal mentions for fields like opinions and problems (leave lists empty if none are mentioned),
and infer overall sentiment and satisfaction score from the user's tone and complaints.
"""

ie_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", ie_system_msg),
        ("human", "Extract details about a customer review of a mobile app, given below.\n{text}"),
    ]
)

Next, we associate the model with the schema and combine the model with the prompt into an IE chain:


In [ ]:
model_name = "openai/gpt-oss-20b"
llm = ChatGroq(model=model_name, temperature=0.0, model_kwargs={'seed':RAND_STATE})

In [ ]:
ie_llm = llm.with_structured_output(schema=AppReview) # this is where the LLM is connected w/ the Schema

ie_chain = ie_prompt_template | ie_llm

Note: the method `with_structured_output` (above) automates the process of binding the schema to the model and parsing the model's output into the structure defined by the schema.

In [ ]:
reviews = [
    "not sure i feel with all the permissions you have to give this app and then you have to create an account and log in makes me believe they are using this app for getting information. it could be done with only the minable permissions and use your own memory.",
    "galaxy s2 love this app. very useful. only thing that bugs me is from time to time my screen blacks out and force closes the app when i back out.",
    "bad. has nothing for end game content and is very very very outdated."
]

Invoking the model produces structured output aligned with the given schema:

In [ ]:
extracted_review_data = [ie_chain.invoke({'text':review}) for review in reviews]

for review, review_data in zip(reviews, extracted_review_data):
  print(review)
  print(f"user's opinion about the app: {review_data.app_opinion}")
  print(f"problems: {review_data.problems}")
  print(f"sentiment: {review_data.sentiment}")
  print(f"satisfaction: {review_data.satisfaction}")
  print("-"*70)

### Combining two schemas

We can define more than one schema and use them for extracting entites of different types - for example, about an event and people attending the event

In [ ]:
from typing import List
from datetime import datetime

class EventAttendee(BaseModel):
    """A person attending the event."""

    name: str = Field(
        description="Name of the attendee.")

    role: Optional[str] = Field(
        default=None,
        description="The attendee's job role or title.")


class EventDetails(BaseModel):
    """Information about a specific event."""

    title: str = Field(description="The official title of the event.")

    date: Optional[str] = Field(
        default=None,
        description="The date of the event in YYYY-MM-DD format.",
        examples=['2025-11-10'])

    location: Optional[str] = Field(
        default=None,
        description="The city and country where the event takes place.")

    attendees: List[EventAttendee] = Field( # note that this field is based on the above defined schema
        default_factory=list,
        description="A list of key people attending the event.")

In [ ]:
event_IE_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert information extraction assistant. Extract details about the event described below."),
        ("human", "Event information: {text_input}"),
    ]
)

event_ie_llm = llm.with_structured_output(schema=EventDetails) # since EventDetails has a reference to EventAttendee, we pass only this scheme

event_ie_chain = event_IE_prompt | event_ie_llm

In [ ]:
input_text = """
We are excited to announce the AI Frontier Summit, scheduled for 2026-03-15 in Tokyo, Japan.
Key speakers include Dr. Emily Carter, the Chief Scientist, and Mr. Alex Chen, the Lead Engineer.
"""

extracted_data = event_ie_chain.invoke({"text_input": input_text})

In [ ]:
type(extracted_data)

Access the extracted data through the fields of the EventDetails class:

In [ ]:
print(f"Event Title: {extracted_data.title}")
print(f"Date: {extracted_data.date if extracted_data.date else 'unknown'}")
print(f"Location: {extracted_data.location if extracted_data.location else 'unknown'}")
print("Attendees:")
for attendie in extracted_data.attendees:
  print(f"\t- {attendie.name}" + f", {attendie.role}" if attendie.role else "")


### Examining the IE chain on a larger data set


To examine in more detail this kind of IE chain in the context of customer reviews we will use a data set of restaurant reviews that originates from Kaggle.com: [10000 Restaurant Reviews](https://www.kaggle.com/datasets/joebeachcapital/restaurant-reviews).<br>

We will load a bunch of restaurant reviews, extract information about what customers liked and disliked, as well as their overall sentiment and compare the latter to human labels.

First, load the data from the file

In [ ]:
import pandas as pd

data = pd.read_csv('data/Restaurant reviews.csv')
data.head(10)

In [ ]:
data.info()

In [ ]:
# drop the columns we will not need
data.drop(columns=['Pictures','7514','Metadata'], inplace=True)

In [ ]:
# drop also rows without a review
data.dropna(subset='Review', inplace=True)
data.reset_index(drop=True, inplace=True)

Compute and examine the review lenght as it does not make much sense to work with overly short reviews

In [ ]:
data['ReviewLen'] = data.Review.apply(len)
data.ReviewLen.describe()

We will exclude overly short reviews (typically not much can be extracted from them, so we should better not waste LLM on them), by setting aside reviews with lenght below 20 characters (~10th percentile).

In [ ]:
# print(len("We will exclude overly short reviews "))

In [ ]:
threshold = 40
data_sub = data.loc[data.ReviewLen > threshold,].copy()
data_sub.reset_index(drop=True, inplace=True)
data_sub.info()

In [ ]:
data_sub.ReviewLen.describe()

We will check the column with customer ratings as its type (str) is not as expected (int or float):

In [ ]:
data_sub.Rating.value_counts()

We'll drop the row with inappropriate value ('Like') and transform numeric ratings from string to float

In [ ]:
data_sub = data_sub.loc[data_sub.Rating != 'Like',]

data_sub.Rating = data_sub.Rating.astype(float)
data_sub.Rating.describe()

To get a better understanding of the data, we'll take a closer look at a random sample of the reviews

In [ ]:
for review in data_sub.Review.sample(10, random_state=RAND_STATE):
  print(review)
  print("-"*70)

Now, define the schema for IE from restaurant reviews

In [ ]:
class RestaurantReview(BaseModel):
    """Information about customer opinions and sentiment expressed in a restaurant review."""

    sentiment: str = Field(
        description="The overall sentiment expressed in the review. Possible options are: positive, negative, neutral, and mixed",
        examples=['positive']
    )

    food_positives: Optional[list] = Field(
        default_factory=list,
        description="Positive opinion(s) and satisfaction expressed in the review about the food served in the restaurant. This field is about food only and no other aspects of the restaurant.",
        examples=[["amazing punjabi food"]]
    )

    other_positives: Optional[list] = Field(
        default_factory=list,
        description="Positive opinion(s) and satisfaction expressed in the review about the restaurant itself, including, for example, its facilites, staff, and atmosphere. This field is about the customer's positive perceptions of various aspects of the restaurant EXCEPT food.",
        examples=[["service was quick", "staff did their best"]]
    )

    food_negatives: Optional[list] = Field(
        default_factory=list,
        description="Negative opinion(s) and dissatisfaction expressed in the review about the food served in the restaurant. This field is about food only and no other aspects of the restaurant.",
        examples=[["stale chicken", "cold roties"]]
    )

    other_negatives: Optional[list] = Field(
        default_factory=list,
        description="Negative opinion(s) and dissatisfaction expressed in the review about the restaurant itself, including, for example, its facilites, staff, and atmosphere. This field is about the customer's negative perceceptions about various aspects of the restaurant EXCEPT food.",
        examples=[["long waiting to be served"]]
    )



We need to instiate an LLM with a different IE schema and somewhat change the prompt:

In [ ]:
system_msg = """
You are an expert information extraction agent, specialised in analysing restaurant reviews posted on the internet and extracting information from them.
Extract literal mentions for fields related to what users liked and disliked about the food and different aspects of the restaurant (leave lists empty if none are mentioned),
and infer the overall sentiment from the user's tone and comments.
"""

restaurant_ie_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", ie_system_msg),
        ("human", "Extract details about a restaurant review, given below:\n{text}"),
    ]
)

llm = ChatGroq(model=model_name, temperature=0.0)
restaurant_ie_llm = llm.with_structured_output(schema=RestaurantReview)

restaurant_ie_chain = restaurant_ie_prompt_template | restaurant_ie_llm

First, try the IE chain on a few reviews only

In [ ]:
import textwrap

restaturant_reviews_sample = data_sub.Review.sample(3, random_state=RAND_STATE).to_list()

for review in restaturant_reviews_sample:
  response = restaurant_ie_chain.invoke({'text': review})
  print(textwrap.fill(review, width=100))
  print()
  print("Extracted info:")
  print(f"Overall sentiment: {response.sentiment}")
  print(f"Positive food-related comments: {response.food_positives}")
  print(f"Negative food-related comments: {response.food_negatives}")
  print(f"Other positive comments: {response.other_positives}")
  print(f"Other negative comments: {response.other_negatives}")
  print("-"*100)

Considering the restrictions on the Groq API use and the limited class time, instead of extracting information from all reviews in our data subset, we will just demonstrate the process on a small sample.

In [ ]:
sample_size = 60

review_sample = data_sub.sample(n=sample_size, random_state=RAND_STATE)
review_sample.info()

In [ ]:
review_sample.head()

Now, we can pass each review to the IE chain and structure the results into a data frame for easier further analysis

In [ ]:
from time import sleep

results = []
errors = []

for i, review in enumerate(review_sample.Review.tolist()):

  if i % 10 == 0: print(f"Processing {i+1}. review")

  try:
    res = restaurant_ie_chain.invoke({"text":review})
    if res:
      res_dict = res.model_dump()
      res_dict['Review'] = review
      results.append(res_dict)
    else: # not expected to occurr, but just in case...
      errors.append({
          'review': review,
          'error': 'unknown error cause'
      })

  except Exception as e:
    print(f"Error when processing review \"{review}\":\n{e}")
    errors.append({
        'review': review,
        'error': str(e)
    })

  sleep(3)


results_df = pd.DataFrame(results)

In [ ]:
results_df.head()

Combine the data from the original data frame with the newly extracted data:

In [ ]:
enriched_review_sample = pd.merge(review_sample[['Review','Restaurant','Rating']], results_df, on='Review', how='left')
enriched_review_sample.head()

Save the 'enriched' data:

In [ ]:
results_fname = 'enriched_sample_of_resturant_reviews.csv'
enriched_review_sample.to_csv(results_fname, index=False)

Let's check the overall distribution of sentiment as assessed by the LLM

In [ ]:
enriched_review_sample.sentiment.value_counts()

The dominant positive sentiment is in line with the high rating values we've seen in the original data.

As a simple form of results check, we'll examine the correspondance between human ratings (`Rating`) and sentiment labels assigned by the LLM (`sentiment`):

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sb

plt.figure(figsize=(9, 5))

sb.kdeplot(data=enriched_review_sample,
           x='Rating',
           hue='sentiment',
           fill=True,
           common_norm=False,
           alpha=0.5,
           linewidth=2)

plt.title('Distribution of restaurant ratings across distinct sentiments')
plt.xlabel('Rating')
plt.ylabel('Density')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

The above plot is aligned with the expecations: positive sentiment co-occurrs with high ratings, negative sentiment with low ratings; mixed is almost non-existant.

As a simple form of internal consistency check of the data extracted by the LLM, we can examine if there is a correspondance between the number of positive and negative items it identified and the sentiment it assigned:

In [ ]:
review_summaries = []

for _, row in enriched_review_sample.iterrows():
  review_summary = {'review':row['Review'], 'sentiment': row['sentiment'], 'positive_cnt':0, 'negative_cnt':0}
  if row['food_positives']: review_summary['positive_cnt'] += len(row['food_positives'])
  if row['other_positives']: review_summary['positive_cnt'] += len(row['other_positives'])
  if row['food_negatives']: review_summary['negative_cnt'] += len(row['food_negatives'])
  if row['other_negatives']: review_summary['negative_cnt'] += len(row['other_negatives'])
  review_summary['simple_diff'] = review_summary['positive_cnt'] - review_summary['negative_cnt']
  review_summaries.append(review_summary)

review_summaries_df = pd.DataFrame(review_summaries)
review_summaries_df.head()

In [ ]:
plt.figure(figsize=(6, 4))

sb.kdeplot(data=review_summaries_df,
           x='simple_diff',
           hue='sentiment',
           fill=True,
           common_norm=False,
           alpha=0.5,
           linewidth=2)

plt.xlabel('Difference of positively and negatively reviewed items')
plt.ylabel('Density')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

This is, again, in line with expectations: higher number of negative (vs positive) items associated with negative sentiment; higher number of positive (vs negative) items associated with positive sentiment.

### To explore further:

#### Knowedge graph extraction

An interesting thing to explore further is the use of LangChain for the extraction of a knowledge graph from the given textual content. This is a more advanced form of information extraction since it includes extracting entities mentioned in the text and their relationships, followed by the creation of a graph out of the extracted information (nodes are the identified entities, whereas edges are the relationships identified among the entities).

For an example how it can be done, check the code and the tutorial at this GitHub repo: [https://github.com/thu-vu92/knowledge-graph-llms/](https://github.com/thu-vu92/knowledge-graph-llms/)

#### LangExtract

In addition, Google's [LangExtract](https://github.com/google/langextract) library is another LLM-based option for IE extraction, which seems to allow for more precise IE than the approach presented here as it is based on an LLM (Gemini) fine-tuned for the IE task and different ways of configuring the task execution.